## Introduction

One of the greatest discoveries of modern astronomy is that the Universe is expanding. Galaxies are observed to recede from one another, with their recession velocities increasing approximately linearly with distance. This empirical relationship, known as **Hubble's Law**, suggests that the Universe itself is evolving with time.

To describe this expansion theoretically, cosmologists begin with **Einstein's General Theory of Relativity**. Under the assumptions that the Universe is homogeneous and isotropic on sufficiently large scales, Einstein's field equations simplify to the **Friedmann equations**.

The Friedmann equation governs the evolution of the cosmic scale factor, $a(t)$, which measures how distances between galaxies change with time. Solving this equation allows us to predict the expansion history of the Universe, estimate its age, and study the influence of matter, radiation, dark energy, and spatial curvature.

For an idealized Universe, the Friedmann equation is

$$
\left(\frac{\dot a}{a}\right)^2
=
H_0^2
\left(
\frac{\Omega_r}{a^4}
+
\frac{\Omega_m}{a^3}
+
\frac{\Omega_k}{a^2}
+
\Omega_\Lambda
\right).
$$

Analytical solutions exist only for a few highly simplified cosmological models. A realistic Universe containing multiple energy components must therefore be solved **numerically**.

In this notebook, we shall

- implement the Friedmann equation in Python,
- solve it numerically using SciPy,
- compute the age of the Universe,
- compare different cosmological models,
- visualize the evolution of the cosmic scale factor
- interpret the resulting expansion histories.

# 1. IMPORTING NECESSARY LIBRARIES

Before solving the Friedmann equation, we import the computational libraries required throughout this notebook.

### NumPy

**NumPy** is the standard numerical computing library in Python. It provides the mathematical functions that are extensively used in scientific computing.

Throughout this notebook, NumPy will be used to

- perform numerical calculations,
- evaluate powers of the scale factor,
- compute square roots

### Matplotlib

**Matplotlib** is Python's standard plotting library.

After solving the Friedmann equation, we shall visualize the evolution of the cosmic scale factor $a(t)$ by plotting it as a function of cosmic time.

### SciPy's `solve_ivp`

The Friedmann equation is a **first-order ordinary differential equation (ODE)**.

In general,

$$
\frac{dy}{dt}=f(t,y).
$$

Instead of attempting to solve this equation analytically, we use SciPy's adaptive Runge–Kutta solver, `solve_ivp`, which computes an accurate numerical approximation to the solution while automatically adjusting its step size to maintain numerical accuracy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# 2. Defining the Cosmological Parameters and Converting Units

The Friedmann equation depends on the present-day composition of the Universe. This composition is described using **density parameters**, denoted by the symbol $\Omega$.

Each density parameter represents the fractional contribution of a particular component relative to the **critical density** of the Universe.

The four components considered in this notebook are

- Matter ($\Omega_m$),
- Radiation ($\Omega_r$),
- Dark Energy ($\Omega_\Lambda$),
- Spatial Curvature ($\Omega_k$).

These satisfy the normalization condition

$$
\Omega_m+\Omega_r+\Omega_\Lambda+\Omega_k=1.
$$

The present-day expansion rate of the Universe is characterized by the **Hubble constant**, denoted by $H_0$.

Observationally,

$$
H_0
=
70
\;
{\rm km\,s^{-1}\,Mpc^{-1}}.
$$

However, our numerical integration measures time in **billions of years (Gyr)**.

Consequently, the Hubble constant must first be converted into units of

$$
{\rm s^{-1}},
$$

using

$$
1~{\rm Mpc}
=
3.086\times10^{19}
~{\rm km},
$$

and subsequently into

$$
{\rm Gyr^{-1}},
$$

using

$$
1~{\rm Gyr}
=
10^9
\times
3.154\times10^7
=
3.154\times10^{16}
~{\rm s}.
$$

These converted quantities allow the differential equation to be solved using cosmic time measured directly in billions of years.

The curvature density parameter is computed from the normalization condition above. Since current observations indicate that the Universe is spatially flat to an excellent approximation, the resulting value of $\Omega_k$ is very close to zero.


In [ ]:
#2) conversion of units for hubble

# Hubble constant now
H0_km_s_Mpc = 70.0  # Hubble constant in km/s/Mpc

#in SI units (s^-1)
H0_per_s = H0_km_s_Mpc / (3.086e19)  # Hubble constant in s^-1

#in Gyr^-1
H0_per_Gyr = H0_per_s * (3.154e16)  # Hubble constant in Gyr^-1

#3) defining density parameters

Omega_m0 = 0.315      # Matter
Omega_r0 = 9.2e-5     # Radiation
Omega_L0 = 0.685      # Dark Energy
Omega_k0 = 1.0 - (Omega_m0 + Omega_r0 + Omega_L0)


# 3. The Friedmann Equation

Having specified the cosmological parameters describing the present-day Universe, we now formulate the equation that governs cosmic expansion.

According to the **Cosmological Principle**, the Universe is homogeneous and isotropic on sufficiently large scales. Under these assumptions, Einstein's field equations reduce to the **Friedmann equations**, which describe the evolution of the cosmic scale factor, $a(t)$.

The first Friedmann equation is

$$
\left(\frac{\dot a}{a}\right)^2
=
H_0^2
\left(
\frac{\Omega_r}{a^4}
+
\frac{\Omega_m}{a^3}
+
\frac{\Omega_k}{a^2}
+
\Omega_\Lambda
\right).
$$

Each term on the right-hand side has a clear physical interpretation.

### Radiation

The radiation density scales as

$$
\rho_r\propto a^{-4}.
$$

The factor of $a^{-3}$ arises because the number density of photons decreases as the Universe expands, while the additional factor of $a^{-1}$ originates from the cosmological redshift, which continuously reduces the energy of every photon.

Consequently,

$$
\rho_r\propto a^{-3}\times a^{-1}=a^{-4}.
$$

---

### Matter

Matter behaves differently.

The number of particles remains constant while the volume increases as

$$
V\propto a^3.
$$

Therefore,

$$
\rho_m\propto a^{-3}.
$$

Unlike photons, matter particles do not lose their rest-mass energy due to cosmological expansion.

---

### Spatial Curvature

The curvature contribution evolves as

$$
\rho_k\propto a^{-2}.
$$

Unlike matter or radiation, this term is purely geometric and arises from the spatial curvature of the Universe.

---

### Dark Energy

The simplest model of dark energy is the cosmological constant.

Its energy density remains constant throughout cosmic history,

$$
\rho_\Lambda=\text{constant},
$$

which is why its contribution appears simply as

$$
\Omega_\Lambda.
$$

As the Universe expands, matter and radiation become progressively diluted, whereas dark energy does not.

Consequently, dark energy eventually dominates the cosmic energy budget, producing the presently observed accelerated expansion.

The quantity appearing on the left-hand side,

$$
\frac{\dot a}{a},
$$

is known as the **Hubble parameter**,

$$
H(t)=\frac{\dot a}{a}.
$$

Unlike the Hubble constant, which describes the expansion rate today, the Hubble parameter varies throughout cosmic history.

Substituting

$$
H=\frac{\dot a}{a}
$$

into the Friedmann equation gives

$$
H(a)
=
H_0
\sqrt{
\frac{\Omega_r}{a^4}
+
\frac{\Omega_m}{a^3}
+
\frac{\Omega_k}{a^2}
+
\Omega_\Lambda
}.
$$

This expression provides the expansion rate of the Universe as a function of the scale factor.

Our objective is to compute how the scale factor evolves with time.

# 4. Converting the Friedmann Equation into an Ordinary Differential Equation

Numerical solvers such as SciPy's `solve_ivp` require the differential equation to be written explicitly in the form

$$
\frac{dy}{dt}=f(t,y).
$$

The Friedmann equation is not immediately written in this form because it contains

$$
\frac{\dot a}{a}.
$$

Using the definition

$$
H=\frac{\dot a}{a},
$$

we multiply both sides by $a$ to obtain

$$
\dot a=aH.
$$

Substituting the Friedmann equation gives

$$
\boxed{
\frac{da}{dt}
=
aH_0
\sqrt{
\frac{\Omega_r}{a^4}
+
\frac{\Omega_m}{a^3}
+
\frac{\Omega_k}{a^2}
+
\Omega_\Lambda
}
}
$$

This is now a first-order ordinary differential equation suitable for numerical integration.

The unknown quantity is the cosmic scale factor $a(t)$, while all cosmological parameters have already been specified.

The next task is to translate this mathematical expression directly into Python.

In [ ]:
def friedmann_equation(t, a, H0):

    a = float(np.atleast_1d(a)[0])

    if a <= 0:
        return 0

    radiation = Omega_r0 / a**4
    matter = Omega_m0 / a**3
    curvature = Omega_k0 / a**2
    dark_energy = Omega_L0

    H_squared = H0**2 * (
        radiation +
        matter +
        curvature +
        dark_energy
    )

    return a * np.sqrt(H_squared)

## Understanding the Python Implementation

The function `friedmann_equation()` is a direct translation of the mathematical equation derived above.

The first three lines

```python
a = float(np.atleast_1d(a)[0])

if a <= 0:
    return 0
```

ensure numerical stability.

During the integration process, SciPy represents the dependent variable as a NumPy array. Since our problem contains only a single variable, we convert this array into a scalar.

The conditional statement prevents numerical errors should the integrator accidentally evaluate the equation at non-physical values of the scale factor.

Next, the contribution of each energy component is computed individually,

```python
radiation = Omega_r0 / a**4 
matter = Omega_m0 / a**3 
curvature = Omega_k0 / a**2 
dark_energy = Omega_L0 
```

This closely mirrors the mathematical form of the Friedmann equation.

Separating these terms instead of writing a single long expression improves readability and makes the physical interpretation immediately clear.

Finally,

```python
return a * np.sqrt(H_squared)
```

implements

$$
\frac{da}{dt}=aH,
$$

which is precisely the differential equation derived in the previous section.

Thus, every line of Python code corresponds directly to one mathematical step in the derivation.

In [ ]:
#5) computing age of universe

a_start = 1e-5  #(just a proxy of 0)
a_today = 1.0


def dt_da(a, t, H0):

    dadt = friedmann_equation(t, a, H0)

    if dadt <= 0: #(should mean before/ at the big bang)
        return 0  

    return 1.0 / dadt


age_solution = solve_ivp(
    fun=lambda a, t: dt_da(a, t, H0_per_Gyr),
    t_span=(a_start, a_today),
    y0=[0],
    t_eval=np.linspace(a_start, a_today, 500)
)

age_of_universe = age_solution.y[0][-1]

print(f"Age of Universe = {age_of_universe:.2f} Gyr")

# 6. Exploring Different Cosmological Models

Thus far, we have solved the Friedmann equation using the cosmological parameters corresponding to the observed Universe. However, one of the greatest strengths of numerical simulations is that they allow us to explore **hypothetical universes** by simply changing the density parameters.

The Friedmann equation,

$$
\left(\frac{\dot a}{a}\right)^2
=
H_0^2
\left(
\frac{\Omega_r}{a^4}
+
\frac{\Omega_m}{a^3}
+
\frac{\Omega_k}{a^2}
+
\Omega_\Lambda
\right),
$$

depends entirely on the relative contributions of matter, radiation, curvature, and dark energy. By modifying these parameters, we can investigate how different physical ingredients influence the expansion history of the Universe.

In this notebook, we compare three representative cosmological models.

---

## (i) The matter-radiation-dark energy Universe (Our Universe)

The first model corresponds to the currently accepted cosmological model, commonly known as the **Lambda Cold Dark Matter ($\Lambda$CDM) model**.

Its present-day density parameters are

$$
\Omega_m = 0.315,
\qquad
\Omega_\Lambda = 0.685,
\qquad
\Omega_r \approx 0,
\qquad
\Omega_k \approx 0.
$$

This model closely reproduces observations of the Cosmic Microwave Background, large-scale structure, and distant Type Ia supernovae.

During the early Universe, matter dominates the expansion, causing gravity to slow the rate of expansion. As the Universe grows larger, the matter density decreases while the dark energy density remains constant. Eventually, dark energy becomes the dominant component, leading to the accelerated expansion observed today.

---

## (ii) Matter-Dominated Universe (Einstein–de Sitter Model)

The second model is a Universe containing only matter.

Its density parameters are

$$
\Omega_m = 1,
\qquad
\Omega_\Lambda = 0,
\qquad
\Omega_r = 0,
\qquad
\Omega_k = 0.
$$

Historically, this model was widely studied before the discovery of cosmic acceleration.

Since gravity is the only significant influence, the expansion continually slows with time. Unlike the $\Lambda$CDM Universe, there is no dark energy to drive accelerated expansion.

Consequently, the scale factor grows more slowly than in the observed Universe.

---

## (iii) Empty Universe (Milne Universe)

The final model represents an idealized Universe containing neither matter nor dark energy.

Its density parameters are

$$
\Omega_m = 0,
\qquad
\Omega_\Lambda = 0,
\qquad
\Omega_r = 0,
\qquad
\Omega_k = 1.
$$

In this case, there is no matter to produce gravitational attraction and no dark energy to accelerate the expansion.

The expansion is governed entirely by the geometry of spacetime. As a result, the scale factor increases approximately linearly with time,

$$
a(t)\propto t.
$$

Although this model does not describe our Universe, it provides an important theoretical reference against which more realistic cosmological models can be compared.

---

By solving the Friedmann equation for each of these three models, we can directly visualize how the composition of the Universe determines its expansion history. This comparison illustrates one of the central ideas of modern cosmology: **the evolution of the Universe is governed not merely by the existence of expansion, but by what the Universe is made of.**

In [ ]:
#6) "a" for different universes

time = np.linspace(0, 30, 1000)

def solve_universe(Omega_m, Omega_L, Omega_r):

    global Omega_m0, Omega_L0, Omega_r0, Omega_k0

    Omega_m0 = Omega_m
    Omega_L0 = Omega_L
    Omega_r0 = Omega_r
    Omega_k0 = 1.0 - (Omega_m0 + Omega_r0 + Omega_L0)

    solution = solve_ivp(
        fun=lambda t, a: friedmann_equation(t, a, H0_per_Gyr),
        t_span=(0, 30),
        y0=[1e-5],
        t_eval=time
    )

    return solution.t, solution.y[0]


# Benchmark (Our Universe)
t_LCDM, a_LCDM = solve_universe(0.315, 0.685, 0)

# Matter-only Universe
t_M, a_M = solve_universe(1.0, 0.0, 0.0)

# Empty (Milne) Universe
t_E, a_E = solve_universe(0.0, 0.0, 0.0)

#7) plotting the results

plt.figure(figsize=(10,6))

plt.plot(
    t_LCDM,
    a_LCDM,
    lw=2,
    label="Our Universe ($\\Omega_m=0.315$, $\\Omega_\\Lambda=0.685$)"
)

plt.plot(
    t_M,
    a_M,
    '--',
    lw=2,
    label="Matter-only Universe"
)

plt.plot(
    t_E,
    a_E,
    ':',
    lw=2,
    label="Empty Universe"
)

plt.axhline(
    1,
    color='gray',
    linestyle='--',
    alpha=0.6,
    label='Today ($a=1$)'
)

plt.axvline(
    age_of_universe,
    color='blue',
    linestyle='-.',
    alpha=0.7,
    label=f'Age = {age_of_universe:.2f} Gyr'
)

plt.xlim(0,30)
plt.ylim(0,2.5)

plt.xlabel("Cosmic Time (Gyr)")
plt.ylabel("Scale Factor  a(t)")
plt.title("Evolution of the Cosmic Scale Factor")

plt.grid(alpha=0.3)
plt.legend()

plt.show()
